In [1]:
from pathlib import Path
import json
import os

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

In [2]:
PROJECT_ROOT = Path('.').resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'liar_binary.csv'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'

for path in [MODELS_DIR, REPORTS_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

Use the original LIAR train/valid/test split to avoid accidental data leakage between training and evaluation.

In [3]:
df = pd.read_csv(DATA_PATH)

train_df = df[df['split'] == 'train'].copy()
valid_df = df[df['split'] == 'valid'].copy()
test_df = df[df['split'] == 'test'].copy()

X_train = train_df['statement'].fillna('')
y_train = train_df['label']

X_valid = valid_df['statement'].fillna('')
y_valid = valid_df['label']

X_test = test_df['statement'].fillna('')
y_test = test_df['label']

label_names = ['not_credible', 'credible']

In [4]:
def evaluate_binary_model(name: str, y_true, y_pred) -> dict[str, float | str]:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average='binary',
        pos_label=1,
        zero_division=0,
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average='macro',
        zero_division=0,
    )
    return {
        'model': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_credible': precision,
        'recall_credible': recall,
        'f1_credible': f1,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
    }

In [5]:
def save_confusion_matrix(y_true, y_pred, title: str, output_path: Path) -> None:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    display.plot(values_format='d')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.show()

In [6]:
metrics_rows = []
prediction_frames = []

In [7]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)

In [8]:
metrics_rows.append(evaluate_binary_model('Dummy baseline', y_test, dummy_pred))
print(classification_report(y_test, dummy_pred, target_names=label_names, zero_division=0))

save_confusion_matrix(
    y_test,
    dummy_pred,
    'Dummy baseline confusion matrix',
    FIGURES_DIR / 'confusion_matrix_dummy.png',
)

              precision    recall  f1-score   support

not_credible       0.55      1.00      0.71       553
    credible       0.00      0.00      0.00       449

    accuracy                           0.55      1002
   macro avg       0.28      0.50      0.36      1002
weighted avg       0.30      0.55      0.39      1002



/var/folders/y0/z0x4blhs1hs31p1__yb1v57m0000gn/T/ipykernel_85042/111386560.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
joblib.dump(dummy, MODELS_DIR / 'dummy_baseline.joblib')
print(f"Sanity check! Dummy model file was succesfully saved: {os.path.exists(MODELS_DIR / 'dummy_baseline.joblib')}")

Sanity check! Dummy model file was succesfully saved: True


The majority-class dummy model is a minimum reference point and helps verify whether later models learn any claim-level signal beyond class imbalance.

TF-IDF converts each statement into weighted word and phrase features. Logistic Regression learns which phrases are more associated with credible or not credible statements. For a given prediction, the most influential phrases are approximated by coefficient x TF-IDF value.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=30_000,
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=1.0,
    )),
])

model_tfidf.fit(X_train, y_train)
tfidf_pred = model_tfidf.predict(X_test)

In [11]:
metrics_rows.append(evaluate_binary_model("TF-IDF + Logistic Regression", y_test, tfidf_pred))
print(classification_report(y_test, tfidf_pred, target_names=label_names, zero_division=0))

save_confusion_matrix(
    y_test,
    tfidf_pred,
    "TF-IDF Logistic Regression confusion matrix",
    FIGURES_DIR / "confusion_matrix_tfidf.png",
)

              precision    recall  f1-score   support

not_credible       0.65      0.60      0.63       553
    credible       0.55      0.61      0.58       449

    accuracy                           0.60      1002
   macro avg       0.60      0.60      0.60      1002
weighted avg       0.61      0.60      0.61      1002



/var/folders/y0/z0x4blhs1hs31p1__yb1v57m0000gn/T/ipykernel_85042/111386560.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
joblib.dump(model_tfidf, MODELS_DIR / "tfidf_logreg.joblib")

['/Users/marcinmalek/Documents/Codes/Projects/WORK/news-credibility/models/tfidf_logreg.joblib']

In [13]:
vectorizer = model_tfidf.named_steps["tfidf"]
clf = model_tfidf.named_steps["clf"]
feature_names = vectorizer.get_feature_names_out()
coefs = clf.coef_[0]

coef_df = pd.DataFrame({
    "ngram": feature_names,
    "coefficient": coefs,
})

credible_terms = (
    coef_df.sort_values("coefficient", ascending=False)
    .head(20)
    .assign(direction="credible")
)
not_credible_terms = (
    coef_df.sort_values("coefficient", ascending=True)
    .head(20)
    .assign(direction="not_credible")
)

top_ngrams = pd.concat([credible_terms, not_credible_terms], ignore_index=True)
top_ngrams.to_csv(REPORTS_DIR / "tfidf_top_ngrams.csv", index=False)
top_ngrams

,ngram,coefficient,direction
0,percent,2.519287,credible
1,day,1.990772,credible
2,georgia,1.913797,credible
3,countries,1.753492,credible
4,half,1.699468,credible
5,times,1.641379,credible
6,average,1.633933,credible
7,months,1.574807,credible
8,million,1.565574,credible
9,terms,1.541399,credible


In [14]:
def explain_tfidf_prediction(text: str, top_n: int = 10) -> pd.DataFrame:
    transformed = vectorizer.transform([text])
    row = transformed.tocoo()
    contributions = []
    for _, feature_idx, tfidf_value in zip(row.row, row.col, row.data):
        contributions.append({
            "ngram": feature_names[feature_idx],
            "tfidf_value": tfidf_value,
            "coefficient": coefs[feature_idx],
            "contribution": tfidf_value * coefs[feature_idx],
        })
    return (
        pd.DataFrame(contributions)
        .sort_values("contribution", key=lambda s: s.abs(), ascending=False)
        .head(top_n)
    )

example_indices = test_df.sample(3, random_state=42).index
explanation_rows = []

for idx in example_indices:
    text = test_df.loc[idx, "statement"]
    pred = int(model_tfidf.predict([text])[0])
    explanation = explain_tfidf_prediction(text, top_n=8)
    for _, row in explanation.iterrows():
        explanation_rows.append({
            "statement_id": test_df.loc[idx, "statement_id"],
            "statement": text,
            "true_label": int(test_df.loc[idx, "label"]),
            "predicted_label": pred,
            **row.to_dict(),
        })

pd.DataFrame(explanation_rows).to_csv(REPORTS_DIR / "tfidf_example_explanations.csv", index=False)

This model uses a pretrained sentence-transformer to convert each short claim into dense semantic embeddings. A lightweight Logistic Regression classifier is trained on top of those embeddings. This satisfies the transformer/embedding-based part of the MVP without fine-tuning a large transformer.

In [15]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

X_train_emb = embedding_model.encode(X_train.tolist(), show_progress_bar=True, normalize_embeddings=True)
X_valid_emb = embedding_model.encode(X_valid.tolist(), show_progress_bar=True, normalize_embeddings=True)
X_test_emb = embedding_model.encode(X_test.tolist(), show_progress_bar=True, normalize_embeddings=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/254 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [16]:
embedding_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        C=1.0,
    )),
])

embedding_clf.fit(X_train_emb, y_train)
embedding_pred = embedding_clf.predict(X_test_emb)

In [17]:
metrics_rows.append(evaluate_binary_model("Sentence embeddings + Logistic Regression", y_test, embedding_pred))
print(classification_report(y_test, embedding_pred, target_names=label_names, zero_division=0))

save_confusion_matrix(
    y_test,
    embedding_pred,
    "Sentence embeddings confusion matrix",
    FIGURES_DIR / "confusion_matrix_embeddings.png",
)

              precision    recall  f1-score   support

not_credible       0.68      0.62      0.65       553
    credible       0.58      0.64      0.61       449

    accuracy                           0.63      1002
   macro avg       0.63      0.63      0.63      1002
weighted avg       0.64      0.63      0.63      1002



/var/folders/y0/z0x4blhs1hs31p1__yb1v57m0000gn/T/ipykernel_85042/111386560.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
joblib.dump(
    {
        "embedding_model_name": EMBEDDING_MODEL_NAME,
        "classifier": embedding_clf,
        "notes": "Reload SentenceTransformer(embedding_model_name), encode statements, then call classifier.predict(...).",
    },
    MODELS_DIR / "transformer_or_embeddings.joblib",
)

['/Users/marcinmalek/Documents/Codes/Projects/WORK/news-credibility/models/transformer_or_embeddings.joblib']

BERTopic is used as a topic-level credibility signal, not as a conventional supervised classifier. It discovers topics in the training statements, then each topic receives an empirical not-credible risk score based on training labels.

In [19]:
from bertopic import BERTopic

bertopic_model = BERTopic(
    language="english",
    calculate_probabilities=False,
    verbose=True,
)

train_topics, _ = bertopic_model.fit_transform(X_train.tolist())

2026-05-29 15:47:43,543 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/254 [00:00<?, ?it/s]

2026-05-29 15:47:53,938 - BERTopic - Embedding - Completed ✓


2026-05-29 15:47:53,938 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


2026-05-29 15:48:05,454 - BERTopic - Dimensionality - Completed ✓


2026-05-29 15:48:05,454 - BERTopic - Cluster - Start clustering the reduced embeddings


2026-05-29 15:48:05,628 - BERTopic - Cluster - Completed ✓


2026-05-29 15:48:05,631 - BERTopic - Representation - Fine-tuning topics using representation models.


2026-05-29 15:48:05,719 - BERTopic - Representation - Completed ✓


In [20]:
train_topic_df = train_df.copy()
train_topic_df["topic"] = train_topics

topic_risk = (
    train_topic_df
    .groupby("topic")
    .agg(
        n_train_examples=("statement", "size"),
        p_credible=("label", "mean"),
    )
    .reset_index()
)
topic_risk["p_not_credible"] = 1 - topic_risk["p_credible"]
topic_risk["predicted_topic_label"] = np.where(
    topic_risk["p_not_credible"] >= 0.5,
    "not_credible",
    "credible",
)

In [21]:
def topic_keywords(topic: int, top_n: int = 10) -> str:
    if topic == -1:
        return "outlier/noise"
    words = bertopic_model.get_topic(topic) or []
    return ", ".join(word for word, _ in words[:top_n])


def topic_examples(topic: int, n: int = 3) -> str:
    examples = (
        train_topic_df[train_topic_df["topic"] == topic]["statement"]
        .head(n)
        .tolist()
    )
    return " || ".join(examples)


topic_risk["topic_keywords"] = topic_risk["topic"].apply(topic_keywords)
topic_risk["example_statements"] = topic_risk["topic"].apply(topic_examples)

topic_risk = topic_risk[
    [
        "topic",
        "topic_keywords",
        "n_train_examples",
        "p_not_credible",
        "predicted_topic_label",
        "example_statements",
    ]
]

topic_risk.to_csv(REPORTS_DIR / "bertopic_topic_risk.csv", index=False)
topic_risk.head()

,topic,topic_keywords,n_train_examples,p_not_credible,predicted_topic_label,example_statements
0,-1,outlier/noise,3065,0.541272,not_credible,The Chicago Bears have had more starting quart...
1,0,"health, care, obamacare, medicare, insurance, ...",730,0.679452,not_credible,Health care reform legislation is likely to ma...
2,1,"wisconsin, walker, scott, gov, milwaukee, walk...",235,0.689362,not_credible,Two-thirds of Wisconsinites receiving unemploy...
3,2,"never, no, has, not, zero, been, have, any, th...",159,0.578616,not_credible,Mr. Caprio is a career politician who has neve...
4,3,"abortion, planned, parenthood, abortions, wome...",138,0.666667,not_credible,Says the Annies List political group supports ...


In [22]:
test_topics, _ = bertopic_model.transform(X_test.tolist())

default_risk = 1 - y_train.mean()
topic_risk_dict = dict(zip(topic_risk["topic"], topic_risk["p_not_credible"]))


def predict_from_topic(topic: int, topic_risk_dict: dict[int, float], default_risk: float) -> int:
    risk = topic_risk_dict.get(topic, default_risk)
    return 0 if risk >= 0.5 else 1


bertopic_pred = np.array([
    predict_from_topic(topic, topic_risk_dict, default_risk)
    for topic in test_topics
])

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

2026-05-29 15:48:07,042 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.


2026-05-29 15:48:12,801 - BERTopic - Dimensionality - Completed ✓


2026-05-29 15:48:12,801 - BERTopic - Clustering - Approximating new points with `hdbscan_model`


2026-05-29 15:48:12,825 - BERTopic - Cluster - Completed ✓


In [23]:
metrics_rows.append(evaluate_binary_model("BERTopic topic-risk", y_test, bertopic_pred))
print(classification_report(y_test, bertopic_pred, target_names=label_names, zero_division=0))

save_confusion_matrix(
    y_test,
    bertopic_pred,
    "BERTopic topic-risk confusion matrix",
    FIGURES_DIR / "confusion_matrix_bertopic.png",
)

              precision    recall  f1-score   support

not_credible       0.57      0.83      0.68       553
    credible       0.54      0.24      0.33       449

    accuracy                           0.57      1002
   macro avg       0.56      0.54      0.51      1002
weighted avg       0.56      0.57      0.52      1002



In [24]:
bertopic_model.save(MODELS_DIR / "bertopic_model", serialization="pickle")

2026-05-29 15:48:12,903 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [25]:
top_not_credible_topics = topic_risk.sort_values(
    ["p_not_credible", "n_train_examples"], ascending=[False, False]
).head(10)

top_credible_topics = topic_risk.sort_values(
    ["p_not_credible", "n_train_examples"], ascending=[True, False]
).head(10)

display(top_not_credible_topics)
display(top_credible_topics)

,topic,topic_keywords,n_train_examples,p_not_credible,predicted_topic_label,example_statements
119,118,"prayer, prayed, prayers, weinstein, casey, rug...",11,1.000000,not_credible,"Prior to 1962, everybody prayed before school ..."
112,111,"muslim, muslims, obama, chinese, name, mohamme...",12,0.916667,not_credible,59 percent of Americans today believe that Bar...
66,65,"benghazi, attack, libya, rice, requests, attac...",23,0.869565,not_credible,The amount of attention paid this week to Chri...
95,94,"borders, hillary, open, clinton, refugees, pou...",15,0.866667,not_credible,Says Hillary Clinton wants to have open border...
52,51,"isis, islamic, terrorist, alqaida, radical, gr...",28,0.857143,not_credible,Says the man who rushed the stage at him in Da...
26,25,"raise, obama, taxes, tax, barack, 42000, trans...",47,0.851064,not_credible,The jobs bill includes President Obamas tax on...
94,93,"amendment, second, 17th, abolish, thomas, repe...",15,0.800000,not_credible,The Fourth Amendment was what we fought the Re...
79,78,"refugees, syrian, refugee, 250000, syria, mand...",19,0.789474,not_credible,"The president said hes going to bring in 250,0..."
98,97,"core, common, data, together, attitudes, finge...",14,0.785714,not_credible,Common Core is not from the federal government...
100,99,"protesters, capitol, windows, crowd, exit, car...",14,0.785714,not_credible,On letting Occupy Atlanta protesters stay in W...


,topic,topic_keywords,n_train_examples,p_not_credible,predicted_topic_label,example_statements
63,62,"ohio, ohios, ohioans, other, jobs, business, c...",24,0.166667,credible,The state of Ohio has one of the lowest unempl...
74,73,"election, seats, republican, votes, party, 180...",22,0.181818,credible,"Before our last three presidents, you have to ..."
67,66,"wealth, bottom, walmart, gap, owns, richest, p...",23,0.217391,credible,"If black America were a country, itd be the 15..."
107,106,"prison, population, incarceration, worlds, ore...",13,0.230769,credible,"[W]e see Americas prison population exploding,..."
89,88,"governors, approval, governor, ratings, rating...",17,0.235294,credible,About half of the presidents have been governo...
37,36,"killed, gunfire, die, died, americans, people,...",37,0.243243,credible,"Since 1968, more Americans have died from gunf..."
25,24,"georgia, georgias, transportation, ranks, elec...",49,0.244898,credible,Electric car sales in Georgia have dropped dra...
93,92,"permit, gun, concealed, florida, purchases, re...",16,0.250000,credible,The Florida Department of Agricultures website...
53,52,"background, checks, check, nra, gun, expanded,...",27,0.259259,credible,"In 1999, the NRA leadership in Washington, pre..."
88,87,"sector, private, privatesector, growth, jobs, ...",17,0.294118,credible,"We created 800,000 new jobs, we cut the unempl..."
